In [ ]:
import time
from base64 import urlsafe_b64encode
from hashlib import sha256
from io import BytesIO
from pathlib import Path
from urllib.parse import urlencode

from aiohttp import ClientSession
from tqdm.auto import tqdm


async def get_user_token(
    sess: ClientSession,
    *,
    secret: str,
    email: str,
) -> str:
    query = urlencode({'userEmail': email, 'timestamp': int(time.time())})
    b64params = urlsafe_b64encode(query.encode()).decode().rstrip('=')
    hash_ = sha256(f'{query}#{secret}'.encode()).hexdigest()

    async with sess.get(f'/api/directs/{b64params}/{hash_}') as rsp:
        rsp.raise_for_status()
        j = await rsp.json()
    return j['user']['token']


async def upload_file(
    sess: ClientSession,
    path: Path,
    *,
    loc_id: int,
    case_id: int,
    chunk_size: int = 1_048_576,
    progress: bool = True,
) -> None:
    async with sess.get(
        '/api/files/open', params={'localizationID': loc_id, 'caseID': case_id}
    ) as rsp:
        rsp.raise_for_status()
        j = await rsp.json()

    params = {'id': j['id'], 'fileName': path.name}
    with (
        tqdm(
            desc=f'uploading {path.name!r}',
            unit_scale=True,
            disable=not progress,
        ) as tq,
        path.open('rb') as fp,
    ):
        while data := fp.read(chunk_size):
            end = fp.tell()
            async with sess.post(
                '/api/files/upload',
                params=params | {'start': tq.n, 'end': end},
                data={'file': BytesIO(data)},
            ) as rsp:
                rsp.raise_for_status()
                await rsp.json()

            tq.update(end - tq.n)

        async with sess.get('/api/files/close', params=params) as rsp:
            rsp.raise_for_status()
            await rsp.json()


base_url = 'http://bs03.wss.local/'
# base_url = 'http://localhost:5605/'
# base_url = 'https://dev2.pathvision.ai/'
# base_url = 'http://localhost:8080/'
secret = 'Secret PHRASE should be kept In $ecR3T'
email = 'pv@pv.pv'

async with ClientSession(base_url) as sess:
    token = await get_user_token(sess, email=email, secret=secret)
    sess.headers.add('Authorization', f'Bearer {token}')
    await upload_file(
        sess,
        Path('D:/datasets/ckb/_test/JP2K-33003-1.svs'),
        loc_id=0,
        case_id=1969,
        # progress=False,
    )

upload 'JP2K-33003-1.svs': 0.00it [00:00, ?it/s]